# VeriVision / Camera-AI: YOLOv8 Training Pipeline
This notebook contains the complete training pipeline used by the VeriVision backend. You can use this to manually train and visualize models in Google Colab before deploying them to your application.

**Make sure to enable the T4 GPU in Colab: Runtime > Change runtime type > Hardware accelerator > T4 GPU.**

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

## 1. Unzip Dataset
Upload your dataset zip file to the Colab environment first.

In [ ]:
import zipfile
import os

dataset_zip = 'dataset.zip' # <-- Change this to your uploaded zip filename
extract_dir = 'dataset'

if os.path.exists(dataset_zip):
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f'Successfully extracted to ./{extract_dir}/')
else:
    print(f'Please upload {dataset_zip} to the Colab environment using the file browser on the left.')

## 2. Visualize Sample Data
Let's look at a few random images from the dataset.

In [ ]:
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image

# Find all JPG/PNG images in the extracted dataset
image_paths = glob.glob(f'{extract_dir}/**/*.jpg', recursive=True)
image_paths.extend(glob.glob(f'{extract_dir}/**/*.png', recursive=True))

if image_paths:
    # Pick 6 random images
    sample_paths = random.sample(image_paths, min(6, len(image_paths)))
    
    plt.figure(figsize=(15, 10))
    for i, img_path in enumerate(sample_paths):
        plt.subplot(2, 3, i + 1)
        img = Image.open(img_path)
        plt.imshow(img)
        # Print the folder name as the class label
        class_label = os.path.basename(os.path.dirname(img_path))
        plt.title(f'Class: {class_label}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No images found in the dataset directory.')

## 3. Train the Model
Train the YOLOv8 model on the dataset. Adjust `task_type` depending on whether you are doing classification or detection.

In [ ]:
from ultralytics import YOLO

task_type = 'classification' # or 'detection'

if task_type == 'classification':
    model = YOLO('yolov8n-cls.pt')
    data_path = f'{extract_dir}' # Root folder containing train/ and valid/ folders
else:
    model = YOLO('yolov8n.pt')
    data_path = f'{extract_dir}/data.yaml' # YAML file for detection datasets

# Start training (equivalent to the backend training loop)
results = model.train(
    data=data_path,
    epochs=25,
    imgsz=224,
    batch=16,
    lr0=0.001,
    optimizer='Adam',
    project='runs',
    name='verivision_model',
    exist_ok=True,
    patience=10
)

## 4. Training Visualizations
YOLOv8 automatically saves loss curves, confusion matrices, and metrics in the project directory.

In [ ]:
# 1. Plot Training Results (Loss & Accuracy curves)
results_img = 'runs/verivision_model/results.png'
if os.path.exists(results_img):
    print("--- Training Metrics (Loss & Accuracy) ---")
    img = Image.open(results_img)
    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# 2. Plot Confusion Matrix
cm_img = 'runs/verivision_model/confusion_matrix.png'
if os.path.exists(cm_img):
    print("\n--- Confusion Matrix ---")
    img = Image.open(cm_img)
    plt.figure(figsize=(12, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 5. Evaluate and Visualize Test Data
Run the freshly trained model on a few validation images to see its real-world performance and confidence scores.

In [ ]:
# Get some validation images
val_images = glob.glob(f'{extract_dir}/**/val*/**/*.jpg', recursive=True)
if not val_images:
    val_images = glob.glob(f'{extract_dir}/**/*.jpg', recursive=True)

if val_images:
    test_paths = random.sample(val_images, min(4, len(val_images)))
    
    # Run Inference
    predict_results = model(test_paths)
    
    # Plot Predictions
    for r in predict_results:
        # YOLO's built-in plot returns a BGR numpy array
        im_array = r.plot()
        # Convert BGR to RGB for matplotlib/PIL
        im = Image.fromarray(im_array[..., ::-1])
        
        plt.figure(figsize=(8, 8))
        plt.imshow(im)
        plt.axis('off')
        plt.show()
else:
    print('No images found to run inference on.')